# Practical session: Optimization algorithms — Gradient descent, Newton's method, and applications

**Duration:** 3h

**Objectives:**
- Implement and analyze gradient descent (constant step, then with backtracking line search);
- Implement Newton's method and compare its convergence speed to the gradient methods;
- Apply these algorithms to a machine learning and/or image denoising.

---

## Reminders

We consider the unconstrained minimization problem
$$
\tag{P}\min_{x \in \mathbb{R}^d} f(x),
$$
where $f : \mathbb{R}^d\to \mathbb{R}$ is differentiable.

**Constant-step gradient descent.** We generate the sequence $(x_k)_k$ by
$$
x_{k+1} = x_k -\gamma\nabla f(x_k),
$$
where $\gamma>0$ is the step size and $x_0$ the starting point. If $f$ is convex and $\nabla f$ is $L$-Lipschitz, convergence is guaranteed for $0<\gamma < 2/L$. In particular, if $f\in C^{2}$, one may take
$$
L = \sup_{x}\Vert \nabla^2 f(x)\Vert,
$$
where $\nabla^2 f(x)\in\mathcal{M}_{d,d}(\mathbb{R})$ is the Hessian of $f$ at $x$ and $\Vert\cdot\Vert$ is the spectral norm: $\Vert A\Vert = \sqrt{\lambda_{\max}(A^TA)}$ (for symmetric $A$, $\Vert A \Vert = \max_i |\lambda_i(A)|$).

**Newton's method.** We generate the sequence $(x_k)_k$ by
$$
x_{k+1} = x_k - \left[\nabla^{2}f(x_k)\right]^{-1}\nabla f(x_k).
$$
It requires no step-size choice, but needs computing (and inverting/solving with) the Hessian at every iteration.

---

**Tips:**
- Feel free to reuse a function written in a previous exercise for the next ones (`gradient_descent`, `gradient_descent_backtracking`, `newton_method`).
- Don't hesitate to print the number of iterations and the final cost value: it makes debugging much easier.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (6, 5)


---
## Exercise 1: The quadratic case (warm-up)

We consider the quadratic function
$$
f(x, y) = \frac{1}{2}(a x^2 +b y^2), \qquad a, b > 0.
$$

**Q1.** Define the Python oracles `f_quad(x, a, b)` and `grad_f_quad(x, a, b)` (with `x = np.array([x0, x1])`), i.e. functions that return respectively the value of $f$ and its gradient $\nabla f$.

> **Tip.** For this quadratic $f$, the Hessian $\nabla^2 f(x) = \begin{pmatrix} a & 0 \\ 0 & b\end{pmatrix}$ is **constant** (it does not depend on $x$), so $L = \sup_x \Vert \nabla^2 f(x)\Vert = \max(a,b)$ is immediate to compute — no need for a `sup` over anything. This will no longer be the case for the Rosenbrock function in Exercise 2.


In [ ]:
def f_quad(x, a, b):
    """TO COMPLETE: return f(x) = 0.5*(a*x0^2 + b*x1^2)"""
    # ...


def grad_f_quad(x, a, b):
    """TO COMPLETE: return the gradient of f at x, as a np.array of size 2"""
    # ...


**Q2.** Implement the function `gradient_descent(f, grad_f, x0, gamma, epsilon, max_iter=10000)` that performs constant-step gradient descent.
- **Stopping criterion:** stop as soon as $\|\nabla f(x_k)\| \le \epsilon$ (or after `max_iter` iterations).
- **Outputs:** the final point `x_min`, an array `history_cost` containing the values $f(x_k)$, and an array `history_points` containing the iterates $x_k$.

In [1]:
def gradient_descent(f, grad_f, x0, gamma, epsilon, max_iter=10000):
    """TO COMPLETE."""

**Q3.** Test the algorithm with $x_0 = (2, 5)$, $a = 1$, $b = 8$ and $\gamma = 0.1$. Plot:
- the cost value $f(x_k)$ as a function of $k$ (log scale on the y-axis, using `plt.semilogy`);
- the iterates $x_k$ superimposed on the level curves of $f$.


In [ ]:
# TO COMPLETE

**Q4. Analysis.** We want to observe the effect of the step size, the starting point, and the conditioning of the problem. Redo the previous questions while varying:

**a)** the step size $\gamma$ (with $a=1, b=8, x_0=(2,5)$ fixed). Compare for instance $\gamma \in \{0.05,\ 0.24,\ 0.3\}$. Recall the theoretical bound $\gamma_{\max} = 2/L$ with $L = \max(a,b)$, and comment on what you observe for each of the three values (slow / fast convergence, or divergence).

**b)** several starting points $x_0$ (with a reasonable $\gamma$ fixed).

**c)** different pairs $(a,b)$, in particular increasingly ill-conditioned pairs (the condition number is $\kappa = \max(a,b)/\min(a,b)$). What do you notice about the number of iterations needed to converge as $\kappa$ increases?


In [ ]:
# TO COMPLETE: effect of the step size gamma


---
## Exercise 2: The Rosenbrock function (a non-convex problem)

We consider the Rosenbrock function, which has a unique global minimum at $(1,1)$, with value $g(1,1)=0$:
$$
g(x,y) = 100(y-x^2)^2 + (1-x)^2.
$$

Its gradient is
$$
\nabla g(x, y) = \begin{pmatrix} -400x(y-x^2) - 2(1-x) \\ 200(y-x^2) \end{pmatrix}.
$$

**Q1.** Implement the oracles `f_rosen(x)` and `grad_f_rosen(x)`.


In [ ]:
def f_rosen(x):
    # TO COMPLETE
    


def grad_f_rosen(x):
    # TO COMPLETE
    


**Q2.** Test `gradient_descent` on $g$ starting from $x_0 = (2, 5)$ with the small step $\gamma = 0.002$, then with smaller steps ($\gamma = 0.001$, $\gamma = 0.0005$). What happens with $\gamma=0.002$? Also try starting from a point closer to the minimum, e.g. $x_0=(-1,1)$, with these same step sizes. Comment.


In [ ]:
# TO COMPLETE


**Q3. Backtracking line search.** Implement `gradient_descent_backtracking(f, grad_f, x0, gamma, alpha, beta, epsilon, max_iter=10000)`.

- **Armijo condition:** while the current step $t$ (initialized at $\gamma$) does not satisfy
$$
f(x_k - t \nabla f(x_k)) \le f(x_k) - \alpha\, t \|\nabla f(x_k)\|^2,
$$
reduce $t \leftarrow \beta t$, then set $x_{k+1} = x_k - t\nabla f(x_k)$.
- Use $\gamma = 2$ (initial step), $\alpha=0.25$ and $\beta=0.5$.
- Also count the **total number of calls to $f$** (one for each evaluation, including the ones inside the `while` loop that get rejected) and print it alongside the iteration count.

Test on $g$ from $x_0=(2,5)$, plot the trajectory of the iterates over the level curves, and compare the number of *outer iterations* **and** the number of *calls to $f$* to those of Q2 (where constant-step gradient descent makes exactly one call to $f$ per iteration, for the stopping test / history only — the update itself doesn't need $f$ at all).


In [2]:
def gradient_descent_backtracking(f, grad_f, x0, gamma, alpha, beta, epsilon, max_iter=10000):
    """TO COMPLETE."""

---
## Exercise 3: Newton's method

Recall Newton's method:
$$
x_{k+1} = x_k - [\nabla^{2}f(x_k)]^{-1}\nabla f(x_k).
$$

The Hessian of the Rosenbrock function is
$$
\nabla^2 g(x, y) = \begin{pmatrix} 1200x^2 - 400y + 2 & -400x \\ -400x & 200 \end{pmatrix}.
$$

**Q1.** Implement the oracle `hess_f_rosen(x)`, then the function `newton_method(f, grad_f, hess_f, x0, epsilon, max_iter=1000)` that returns `(x_min, history_cost, history_points)`.

> **Tip — never invert a matrix to solve a linear system.** To get the Newton direction $d$ solving $\nabla^2f(x_k)\, d = -\nabla f(x_k)$, use `d = np.linalg.solve(H, -g)` rather than `d = -np.linalg.inv(H) @ g`. The two are mathematically equivalent, but `solve` is both cheaper (it factorizes $H$ once, roughly $O(d^3)$, instead of computing a full inverse and then a matrix-vector product) and more numerically stable (explicit inversion can amplify rounding errors badly when $H$ is close to singular or ill-conditioned). This is a general rule in numerical linear algebra, not specific to this exercise: *"never compute a matrix inverse just to multiply it by a vector."*


In [3]:
def hess_f_rosen(x):
    """TO COMPLETE."""
  


def newton_method(f, grad_f, hess_f, x0, epsilon, max_iter=1000):
    """TO COMPLETE."""

**Q2.** Test `newton_method` on $g$ (Rosenbrock) from $x_0=(2,5)$, then on $f$ (Exercise 1, with $a=1,b=8$) from $x_0=(2,5)$. How many iterations are needed in the quadratic case? Why (justify in one sentence, from the formula of Newton's method)?

In [ ]:
# TO COMPLETE


**Q3. Comparing the three methods.** On the Rosenbrock function, plot on the same graph (log scale on the y-axis) the evolution of $g(x_k)$ for:
- constant-step gradient descent (with a stable step, e.g. $\gamma=0.001$);
- gradient descent with backtracking;
- Newton's method;

all starting from $x_0=(2,5)$. Comment on the relative convergence speed of the three methods.


In [ ]:
# TO COMPLETE


---
## Application : Image denoising

We observe a noisy image $u_0 \in \mathbb{R}^{n\times n}$ (a pixel = an intensity value), and we want to reconstruct a denoised version $u$ by solving the Tikhonov regularization problem:
$$
\min_{u\in\mathbb{R}^{n\times n}} \ \underbrace{\tfrac12 \Vert u - u_0\Vert^2}_{\text{data fidelity}} + \underbrace{\tfrac{\lambda}{2}\Vert \nabla u\Vert^2}_{\text{regularization (smoothing)}},
$$
where $\Vert\nabla u\Vert^2 = \sum_{i,j}\big[(u_{i+1,j}-u_{i,j})^2+(u_{i,j+1}-u_{i,j})^2\big]$ (discrete gradient, with Neumann boundary conditions). We can apply gradient descent directly, with
$$
\nabla \Big[\tfrac12\Vert u-u_0\Vert^2 + \tfrac{\lambda}{2}\Vert\nabla u\Vert^2\Big] = (u-u_0) - \lambda\, \Delta u,
$$
where $\Delta u$ is the discrete 5-point Laplacian of $u$ (function `laplacian` provided below). One can show that $\Vert\Delta\Vert \le 8$ (operator norm), so $L \le 1+8\lambda$ is a valid Lipschitz constant.

**Q1.** Reusing your `gradient_descent` function (it works just as well on 2D arrays as on vectors!), denoise the image `u0` below with $\lambda=1$ and $\gamma = 1/(1+8\lambda)$. Display the original, noisy, and denoised images side by side. Also try $\lambda=5$: what happens to the edges of the square?

In [ ]:
def laplacian(u):
    """Discrete 5-point Laplacian, Neumann (reflective) boundary conditions."""
    up = np.pad(u, 1, mode="edge")
    return (up[2:, 1:-1] + up[:-2, 1:-1] + up[1:-1, 2:] + up[1:-1, :-2] - 4 * up[1:-1, 1:-1])


rng = np.random.default_rng(0)
n_img = 24
u_true = np.zeros((n_img, n_img))
u_true[6:18, 6:18] = 1.0
u0 = u_true + 0.3 * rng.standard_normal((n_img, n_img))

plt.imshow(u0, cmap="gray")
plt.title("Noisy image u0")
plt.colorbar()
plt.show()


In [ ]:
def cost_denoise(u, u0, lam):
    # TO COMPLETE: 0.5*||u-u0||^2 + 0.5*lam*||grad u||^2
    


def grad_cost_denoise(u, u0, lam):
    # TO COMPLETE: (u - u0) - lam * laplacian(u)


# TO COMPLETE: call gradient_descent with f = lambda u: cost_denoise(u, u0, lam), etc.
